In [1]:
import sys, pickle
import random as _random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from floorplan_diffusion.data.dataset import (
    ResPlanDataset, MAX_NUM_POINTS, CORNER_IDX_DIMS, ROOM_IDX_DIMS,
    _normalize_keys, ROOM_TYPE_TO_INT,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"
%matplotlib inline

print("Ready.")

Ready.


In [2]:
# Map every type int the encoder can emit back to a name (incl. window=5).
ROOM_TYPE_INT_TO_NAME = {v: k for k, v in ROOM_TYPE_TO_INT.items()}

ROOM_COLORS = {
    "living":     "#E07B54",
    "bedroom":    "#F5C96A",
    "kitchen":    "#A8C5A0",
    "bathroom":   "#88B4D4",
    "balcony":    "#C9A8E0",
    "window":     "#3CB4C4",  # openings we specifically want to track
    "door":       "#888888",
    "front_door": "#444444",
}

OPENINGS = ("door", "front_door", "window")

def decode_plan(arr, cond):
    """Reconstruct per-part (corners, name) from an augmented (arr, cond) pair."""
    coords = arr.T.copy()
    real_mask = cond["src_key_padding_mask"] < 0.5
    room_type_oh = cond["room_types"]
    room_idx_oh  = cond["room_indices"]

    rooms = {}
    for i in np.where(real_mask)[0]:
        rtype_int = int(np.argmax(room_type_oh[i]))
        ridx      = int(np.argmax(room_idx_oh[i]))
        rooms.setdefault((ridx, rtype_int), []).append(coords[i])

    return [(np.array(pts), ROOM_TYPE_INT_TO_NAME.get(rtype_int, str(rtype_int)))
            for (ridx, rtype_int), pts in sorted(rooms.items())]


def render_decoded(rooms, title, ax):
    ax.set_title(title, fontsize=9)
    ax.set_aspect("equal")
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
    ax.axhline(0, color="#ccc", lw=0.5); ax.axvline(0, color="#ccc", lw=0.5)
    for pts, name in rooms:
        color = ROOM_COLORS.get(name, "#cccccc")
        if len(pts) >= 3:
            ax.add_patch(plt.Polygon(pts, closed=True, fc=color, ec="black",
                                     alpha=0.6, lw=0.8, label=name))
        ax.scatter(pts[:, 0], pts[:, 1], s=12, c="black", zorder=3)
        # Mark opening centroids so they are easy to follow across rotations.
        if name in OPENINGS:
            cx, cy = pts.mean(axis=0)
            marker_color = {"front_door": "red", "door": "orange", "window": "blue"}[name]
            ax.plot(cx, cy, "x", ms=8, mew=2, color=marker_color, zorder=4)

print("Helpers ready.")

Helpers ready.


In [3]:
PKL_PATH = PROJECT_ROOT / "data" / "raw" / "ResPlan.pkl"

# Load raw plan dicts only — building the full ResPlanDataset materialises a
# 192x192x3 float64 mask stack per plan (~85 MB/plan) and OOMs. We only need one.
with open(PKL_PATH, "rb") as f:
    raw_plans = pickle.load(f)

WINDOW_TYPE_INT     = ROOM_TYPE_TO_INT["window"]      # 5
DOOR_TYPE_INT       = ROOM_TYPE_TO_INT["door"]        # 11
FRONT_DOOR_TYPE_INT = ROOM_TYPE_TO_INT["front_door"]  # 13

# A bare instance so we can call _process_plan without the heavy __init__.
ds = ResPlanDataset.__new__(ResPlanDataset)
ds.set_name = "train"          # augmentation active
ds.num_coords = 2
ds.max_num_points = MAX_NUM_POINTS

RT_END = 2 + 25
PAD_COL = RT_END + CORNER_IDX_DIMS + ROOM_IDX_DIMS

def encoded_type_ints(house_array):
    """Set of room-type ints present among the real (non-padded) points."""
    real = house_array[:, PAD_COL] > 0.5
    return set(np.argmax(house_array[real, 2:RT_END], axis=1).tolist())


In [ ]:
#get the